# 🎧 Project Pulse — Where do *your* songs land?

This is a fun, no-setup tool. It places **your** songs onto a music map built from
sound itself (not streaming behavior), tells you your **taste type**, and shows you
surprising things — like two "same genre" songs landing on opposite ends of the map.

**How to use it:** click **Runtime ▸ Run all** at the top, then follow the prompts to
upload a few songs (or paste a YouTube link). That's it — no installing anything.

> It's a personal experiment, not a verdict on your taste. Have fun with it.

## Step 1 — Set up (installs the audio AI, ~1–2 min)

In [ ]:
#@title Run this to install everything
# Pinned to match the saved model so the frozen map loads correctly.
!pip -q install umap-learn==0.5.12 scikit-learn==1.8.0 joblib==1.5.3 \
    librosa==0.11.0 transformers==5.8.1 plotly==6.7.0 yt-dlp 2>/dev/null
print("Done. If Colab asks you to 'Restart runtime', click it, then Run all again.")

## Step 2 — Load the Pulse map

This downloads the project (the engine + the frozen map). **Set `REPO_URL` to your
GitHub repo** once you've published it.

In [ ]:
#@title Load the project + frozen map
REPO_URL = "https://github.com/mdmmirfan/Project_Pulse.git"  #@param {type:"string"}

import os
if not os.path.exists("Project_Pulse"):
    !git clone -q $REPO_URL Project_Pulse
%cd Project_Pulse

import pandas as pd
import pulse_core

space  = pulse_core.load_reference_space()          # frozen scaler + UMAP + coords
ref     = space["reference_df"]
interp  = pd.read_csv(pulse_core.INTERPRETABLE_CSV)
print(f"Loaded the map: {len(ref)} reference songs.")

## Step 3 — Add your songs

**Option A (easiest):** run the cell and upload 1–5 audio files (`.mp3` / `.wav`).
**Option B:** paste YouTube links instead (set `USE_YOUTUBE = True`).

In [ ]:
#@title Upload audio files OR use YouTube links
from pathlib import Path

USE_YOUTUBE = False  #@param {type:"boolean"}
YOUTUBE_LINKS = ""   #@param {type:"string"}
# ^ if using YouTube, paste links separated by commas

my_files = []
Path("my_songs").mkdir(exist_ok=True)

if USE_YOUTUBE:
    import yt_dlp
    links = [u.strip() for u in YOUTUBE_LINKS.split(",") if u.strip()]
    for url in links:
        opts = {
            "format": "bestaudio/best",
            "outtmpl": "my_songs/%(title)s.%(ext)s",
            "postprocessors": [{"key": "FFmpegExtractAudio",
                                "preferredcodec": "wav"}],
            "quiet": True,
        }
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
        my_files.append("my_songs/" + ydl.prepare_filename(info).split("/")[-1]
                        .rsplit(".", 1)[0] + ".wav")
else:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        dest = Path("my_songs") / name
        dest.write_bytes(uploaded[name])
        my_files.append(str(dest))

print("Got", len(my_files), "song(s):", [Path(f).stem for f in my_files])

## Step 4 — Place your songs on the map ✨

In [ ]:
#@title Analyse + place your songs (loads the AI the first time)
placed = pulse_core.place_audio_files(my_files, space=space, interpretable_ref=interp)
placed[["Track", "taste_type"]]

## Step 5 — Your results

The map below shows my songs (faint cyan = I liked, faint pink = I disliked) and
**your songs as gold diamonds**. Drag to rotate.

In [ ]:
#@title The map
fig = pulse_core.build_overlay_map(ref, placed)
fig.show()

In [ ]:
#@title Your taste type + fun reveals
from pulse_core import INTERPRETABLE_FEATURES, assign_taste_type

# Overall taste type across the songs you added
your_profile = placed[INTERPRETABLE_FEATURES].median().to_dict()
print("YOUR TASTE TYPE:", assign_taste_type(your_profile, interp))
print()

# Which of my songs does each of yours sit closest to?
reveal = pulse_core.nearest_reference(placed, ref)
for _, r in reveal.iterrows():
    print(f"• \"{r['your_song']}\"")
    print(f"    lands right next to my \"{r['nearest_reference']}\" "
          f"(which I marked {r['i_marked_it']})\n")

# Fun: if you added several, which two of YOUR songs are acoustically closest?
if len(placed) >= 2:
    import numpy as np
    xyz = placed[["UX", "UY", "UZ"]].values
    best, pair = 1e9, None
    for i in range(len(placed)):
        for j in range(i + 1, len(placed)):
            d = np.linalg.norm(xyz[i] - xyz[j])
            if d < best:
                best, pair = d, (placed.iloc[i]["Track"], placed.iloc[j]["Track"])
    print(f"Your two most sonically-similar songs: \"{pair[0]}\"  &  \"{pair[1]}\"")

---
### What this means
The map is built from **how songs sound** — their acoustic fingerprint — not from what
other people stream. So songs can land near each other across genres, languages, and
eras. If your favourite and least-favourite songs land close together, that's the
interesting part: taste isn't only about sound. 🎶